# Does a generalized-gamma base unlock covariate effects? (3 pairs)

The two decoupled PR pairs (`PR_following_NMT_3W`, `PR_following_4W`) and
`BTW_following_2W` (n = 73) preferred a generalized gamma in the marginal fit.
Generalized gamma **nests the Weibull** (Weibull = gengamma with a = 1), so this
notebook tests whether the extra shape flexibility lets a covariate attach where the
Weibull could not.

For each pair it (1) fits a gengamma baseline and compares its AIC with the Weibull
baseline, then (2) screens each covariate univariately by letting the AFT scale depend
on it, \(\lambda_i=\exp(\beta_0+\beta_1 z_i)\), with the two gengamma shape parameters
shared. Each covariate gets an LR test against the gengamma baseline, and its gengamma
ΔAIC is placed **next to** the Weibull ΔAIC so `unlocked_by_gengamma` is explicit.

Notes: `loc` is fixed at 0 (keeps the model nested and comparable to the Weibull AFT;
the covariate LR test is valid regardless of where the shared shape sits). Fitting uses
a warm start plus two optimizers and reports convergence. Output → `Tables` folder.

In [1]:
# --- Cell 1: Imports and paths ---
import os, warnings
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.simplefilter("ignore")

BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data2.xlsx")
TABLES    = os.path.join(BASE, "Tables")
os.makedirs(TABLES, exist_ok=True)

TARGETS       = ["BTW_following_2W", "PR_following_NMT_3W", "PR_following_4W"]
MIN_BIN_GROUP = 5
DAIC_MIN      = 2.0

In [2]:
# --- Cell 2: Load data (2W NOT dropped here) and resolve covariates ---
df = pd.read_excel(DATA_PATH).rename(columns={"Time_Headway": "hw"})
df = df[df["Pair"].isin(TARGETS)].copy()

def resolve_share_bus(frame):
    if "share_bus" in frame.columns: return "share_bus"
    hits = [c for c in frame.columns if "bus" in c.lower()]
    return hits[0] if hits else None
BUS_COL = resolve_share_bus(df)

REGISTRY = {
    "speed":      dict(col="Target_Speed_km/hr", type="cont", unit=1,    label="per +1 km/h"),
    "speed_diff": dict(col="Speed_Difference",   type="cont", unit=1,    label="per +1 km/h"),
    "flow":       dict(col="Flow_pcu/hr",        type="cont", unit=1000, label="per +1000 pcu/hr"),
    "off_cen":    dict(col="Off_centeredness",   type="bin",  unit=1,    label="True vs False"),
    "occupancy":  dict(col="Occupancy",          type="bin",  unit=1,    label="True vs False"),
}
if BUS_COL is not None:
    s = pd.to_numeric(df[BUS_COL], errors="coerce")
    REGISTRY["share_bus"] = (dict(col=BUS_COL, type="cont", unit=0.1,  label="per +0.10 share")
                             if s.max() <= 1.5 else
                             dict(col=BUS_COL, type="cont", unit=10.0, label="per +10 (pct pts)"))
CANDIDATES = {k: v for k, v in REGISTRY.items() if v["col"] in df.columns}
print("Pairs:", [p for p in TARGETS if p in df['Pair'].unique()])
print("Covariates:", list(CANDIDATES.keys()))
if BUS_COL: print("share_bus column:", BUS_COL)

Pairs: ['BTW_following_2W', 'PR_following_NMT_3W', 'PR_following_4W']
Covariates: ['speed', 'speed_diff', 'flow', 'off_cen', 'occupancy', 'share_bus']
share_bus column: Share_Bus


In [3]:
# --- Cell 3: Robust fitter + likelihoods (gengamma & Weibull, loc=0) ---
def robust_min(fn, x0, args):
    """Warm start, try Nelder-Mead then Powell, return the better result."""
    best = None
    for method, opts in (("Nelder-Mead", {"maxiter": 20000, "xatol": 1e-8, "fatol": 1e-8}),
                         ("Powell",       {"maxiter": 20000})):
        r = optimize.minimize(fn, x0, args=args, method=method, options=opts)
        if best is None or r.fun < best.fun:
            best = r
    return best

# gengamma: params (log a, c, log scale [+ b1*z])
def gg_null(p, t):
    lp = stats.gengamma.logpdf(t, np.exp(p[0]), p[1], loc=0, scale=np.exp(p[2]))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12
def gg_cov(p, t, z):
    lp = stats.gengamma.logpdf(t, np.exp(p[0]), p[1], loc=0, scale=np.exp(p[2] + p[3] * z))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12
# weibull: params (log c, log scale [+ b1*z])
def wb_null(p, t):
    lp = stats.weibull_min.logpdf(t, np.exp(p[0]), loc=0, scale=np.exp(p[1]))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12
def wb_cov(p, t, z):
    lp = stats.weibull_min.logpdf(t, np.exp(p[0]), loc=0, scale=np.exp(p[1] + p[2] * z))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

In [4]:
# --- Cell 4: Baseline distributions per pair (gengamma vs Weibull) ---
baseline, gg_state, wb_state = [], {}, {}
for pair in [p for p in TARGETS if p in df["Pair"].unique()]:
    t = df.loc[df["Pair"] == pair, "hw"].values
    n = len(t)

    a0, c0, _, s0 = stats.gengamma.fit(t, floc=0)
    rg = robust_min(gg_null, [np.log(a0), c0, np.log(s0)], (t,))
    llg = -rg.fun
    wc0, _, ws0 = stats.weibull_min.fit(t, floc=0)
    rw = robust_min(wb_null, [np.log(wc0), np.log(ws0)], (t,))
    llw = -rw.fun

    aic_gg = 2 * 3 - 2 * llg
    aic_wb = 2 * 2 - 2 * llw
    gg_state[pair] = (rg.x, llg)          # (params, loglik) for covariate warm start
    wb_state[pair] = (rw.x, llw)
    baseline.append(dict(Pair=pair, N=n,
        gg_a=round(np.exp(rg.x[0]), 3), gg_c=round(rg.x[1], 3), gg_scale=round(np.exp(rg.x[2]), 3),
        gg_LogLik=round(llg, 2), gg_AIC=round(aic_gg, 2),
        wb_c=round(np.exp(rw.x[0]), 3), wb_scale=round(np.exp(rw.x[1]), 3), wb_AIC=round(aic_wb, 2),
        gg_minus_wb_AIC=round(aic_gg - aic_wb, 2),      # negative => gengamma better
        gg_better="Yes" if aic_gg < aic_wb else "No"))
baseline_dist = pd.DataFrame(baseline)
baseline_dist

,Pair,N,gg_a,gg_c,gg_scale,gg_LogLik,gg_AIC,wb_c,wb_scale,wb_AIC,gg_minus_wb_AIC,gg_better
0,BTW_following_2W,73,10.604,0.665,0.049,-82.05,170.09,2.349,1.998,171.80,-1.71,Yes
1,PR_following_NMT_3W,59,0.425,5.008,3.941,-86.07,178.13,2.783,3.032,177.29,0.84,No
2,PR_following_4W,43,0.310,7.295,4.381,-62.22,130.43,3.157,3.337,130.42,0.01,No


In [5]:
# --- Cell 5: Screen each covariate under BOTH distributions ---
def strict(p, dAIC):
    return "Yes" if (p < 0.05 and dAIC >= DAIC_MIN) else "No"

rows = []
for pair in [p for p in TARGETS if p in df["Pair"].unique()]:
    g = df[df["Pair"] == pair]
    t = g["hw"].values
    n = len(t)
    (gx, llg0) = gg_state[pair]
    (wx, llw0) = wb_state[pair]

    for name, meta in CANDIDATES.items():
        raw = pd.to_numeric(g[meta["col"]], errors="coerce").values.astype(float)
        if meta["type"] == "cont":
            sd = np.nanstd(raw); z = raw - np.nanmean(raw); ok = sd > 1e-9
        else:
            sd = np.nan; z = raw
            ok = (len(np.unique(raw)) == 2 and min((raw == 0).sum(), (raw == 1).sum()) >= MIN_BIN_GROUP)
        if not ok:
            rows.append(dict(Pair=pair, N=n, Covariate=name, Type=meta["type"],
                             gg_effect="-- low variation --")); continue

        # gengamma + covariate (warm start from gg baseline)
        rg = robust_min(gg_cov, [gx[0], gx[1], gx[2], 0.0], (t, z))
        llg1 = -rg.fun; b1g = rg.x[3]
        LRg = 2 * (llg1 - llg0); pg = stats.chi2.sf(LRg, 1); dAICg = LRg - 2
        # weibull + covariate (warm start from wb baseline)
        rw = robust_min(wb_cov, [wx[0], wx[1], 0.0], (t, z))
        llw1 = -rw.fun; b1w = rw.x[2]
        LRw = 2 * (llw1 - llw0); pw = stats.chi2.sf(LRw, 1); dAICw = LRw - 2

        if meta["type"] == "cont":
            eff = f"{(np.exp(b1g*meta['unit'])-1)*100:+.2f}%  {meta['label']}"
        else:
            eff = f"{(np.exp(b1g)-1)*100:+.2f}%  ({meta['label']})"

        gg_str = strict(pg, dAICg); wb_str = strict(pw, dAICw)
        rows.append(dict(Pair=pair, N=n, Covariate=name, Type=meta["type"],
            gg_effect=eff, gg_coef_b1=round(b1g, 6),
            gg_LR_p="<0.001" if pg < 0.001 else round(pg, 4), gg_dAIC=round(dAICg, 2),
            gg_improves_strict=gg_str,
            wb_LR_p="<0.001" if pw < 0.001 else round(pw, 4), wb_dAIC=round(dAICw, 2),
            wb_improves_strict=wb_str,
            unlocked_by_gengamma="Yes" if (gg_str == "Yes" and wb_str == "No") else "No",
            gg_converged=bool(rg.success)))
screen = pd.DataFrame(rows)
screen

,Pair,N,Covariate,Type,gg_effect,gg_coef_b1,gg_LR_p,gg_dAIC,gg_improves_strict,wb_LR_p,wb_dAIC,wb_improves_strict,unlocked_by_gengamma,gg_converged
0,BTW_following_2W,73,speed,cont,-2.09% per +1 km/h,-0.021090,0.0220,3.25,Yes,0.0088,4.86,Yes,No,True
1,BTW_following_2W,73,speed_diff,cont,-2.67% per +1 km/h,-0.027076,0.0058,5.61,Yes,0.0060,5.55,Yes,No,True
2,BTW_following_2W,73,flow,cont,+4.05% per +1000 pcu/hr,0.000040,0.1247,0.36,No,0.0479,1.91,No,No,True
3,BTW_following_2W,73,off_cen,bin,-12.96% (True vs False),-0.138810,0.2116,-0.44,No,0.4048,-1.31,No,No,True
4,BTW_following_2W,73,occupancy,bin,+0.46% (True vs False),0.004543,0.9782,-2.00,No,0.6728,-1.82,No,No,True
5,BTW_following_2W,73,share_bus,cont,-22.11% per +0.10 share,-2.498747,0.0630,1.46,No,0.1521,0.05,No,No,True
6,PR_following_NMT_3W,59,speed,cont,+0.79% per +1 km/h,0.007862,0.4141,-1.33,No,0.7069,-1.86,No,No,True
7,PR_following_NMT_3W,59,speed_diff,cont,-0.66% per +1 km/h,-0.006630,0.4620,-1.46,No,0.3775,-1.22,No,No,True
8,PR_following_NMT_3W,59,flow,cont,-2.74% per +1000 pcu/hr,-0.000028,0.0565,1.64,No,0.3239,-1.03,No,No,True
9,PR_following_NMT_3W,59,off_cen,bin,-7.62% (True vs False),-0.079269,0.3140,-0.99,No,0.4312,-1.38,No,No,True


In [6]:
# --- Cell 6: Per-pair verdict ---
verdict = []
for pair in [p for p in TARGETS if p in df["Pair"].unique()]:
    b = baseline_dist[baseline_dist["Pair"] == pair].iloc[0]
    sub = screen[(screen["Pair"] == pair) & (screen.get("gg_improves_strict") == "Yes")]
    if len(sub):
        sub = sub.copy(); sub["d"] = pd.to_numeric(sub["gg_dAIC"], errors="coerce")
        top = sub.loc[sub["d"].idxmax()]
        cov_msg = f"{top['Covariate']} (gg_dAIC={top['gg_dAIC']}, p={top['gg_LR_p']})"
        rec = f"gengamma AFT on {top['Covariate']}"
    else:
        cov_msg = "none clears strict rule"
        rec = "keep plain Weibull (no covariate)"
    verdict.append(dict(Pair=pair, N=int(b["N"]),
        gengamma_better_baseline=b["gg_better"],
        covariate_unlocked=cov_msg, recommendation=rec))
verdict = pd.DataFrame(verdict)
verdict

,Pair,N,gengamma_better_baseline,covariate_unlocked,recommendation
0,BTW_following_2W,73,Yes,"speed_diff (gg_dAIC=5.61, p=0.0058)",gengamma AFT on speed_diff
1,PR_following_NMT_3W,59,No,none clears strict rule,keep plain Weibull (no covariate)
2,PR_following_4W,43,No,none clears strict rule,keep plain Weibull (no covariate)


In [7]:
# --- Cell 7: Save to the Tables folder ---
out_path = os.path.join(TABLES, "headway_gengamma_covariate_check.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    baseline_dist.to_excel(xl, sheet_name="Baseline_dist",  index=False)
    screen.to_excel(xl,        sheet_name="Covariate_screen", index=False)
    verdict.to_excel(xl,       sheet_name="Verdict",         index=False)
print("Saved:", out_path)

Saved: D:\Headway\Tables\headway_gengamma_covariate_check.xlsx
